# Experiments

In [148]:
import os
import json
import tempfile
import jsonlines
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

In [149]:
import datetime
import subprocess

RUSE_PATH = "../target/release/Ruse"


def run_ruse(tasks, results_file, log_file, *,
             timeout=datetime.timedelta(hours=1),
             max_iterations=5,
             max_sequence_size=2,
             max_context_depth=3,
             max_memory_usage="110GiB",
             workers_count=64,
             dry_run=False):
    args = [
        "-o", results_file, "--log", log_file,
        "-t", str(int(timeout.total_seconds())),
        "--workers-count", str(workers_count),
        "--max-iterations", str(max_iterations),
        "--max-context-depth", str(max_context_depth),
        "--max-sequence-size", str(max_sequence_size),
        "--max-task-mem", max_memory_usage]
    for task in tasks:
        args.append("-b")
        args.append(task)
    if dry_run:
        args.append("--dry-run")
    subprocess.run([RUSE_PATH, "run"] + args,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def run_experiment(tasks, **kwargs):
    with tempfile.TemporaryDirectory() as temp_dir:
        results_file = os.path.join(temp_dir, "results.json")
        log_file = os.path.join(temp_dir, "log.txt")
        run_ruse(tasks, results_file, log_file, **kwargs)
        with open(results_file, "r") as f:
            results = json.load(f)
        with jsonlines.open(log_file, "r") as reader:
            log = [line for line in reader]

    return results, log

def check_for_parsing_errors(tasks):
    results, log = run_experiment(tasks, dry_run=True)

    for task in results["tasks"]:
        if task["error"] != None:
            print(f"Failed to parse task: {task['path']}")
            print(task["error"])

def get_all_tasks(tasks):
    results, log = run_experiment(tasks, dry_run=True)

    tasks_parsed = {}

    for task in results["tasks"]:
        tasks_parsed[os.path.basename(task["path"])] = [
            task["category"].split(":")[0],
            task["category"].split(":")[1],
            # task["path"],,
        ]

    df = pd.DataFrame.from_dict(tasks_parsed, orient="index", columns=["oop category", "side effects"])
    df.sort_values(by=["oop category", "side effects"], inplace=True)
    return df

def display_tasks(tasks):
    s = tasks.style.set_properties(**{'text-align': 'left'})
    s.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    html_table = s.to_html(justify='left')

    scrollable_html = f"""
<div style="height: 300px; overflow: auto;">
    {html_table}
</div>
"""
    display(HTML(scrollable_html))


### RQ1 - Can Ruse solve synthesis tasks from all categories correctly?

In [150]:
tasks_paths = [
    "../tasks/benchmarks/"
]

tasks = get_all_tasks(tasks_paths)

display_tasks(tasks)
    


,oop category,side effects
binary_search_tree_delete_two_children.sy,Full OOP,With Side Effects
binary_search_tree_unlink_leaf.sy,Full OOP,With Side Effects
set_subtree.sy,Full OOP,With Side Effects
binary_search_tree_valid.sy,Full OOP,Without Side Effects
binary_search_tree_get.sy,Full OOP,Without Side Effects
binary_search_tree_height.sy,Full OOP,Without Side Effects
user_full_name_1_example.sy,Full OOP,Without Side Effects
user_full_name_2.sy,Full OOP,Without Side Effects
user_full_name.sy,Full OOP,Without Side Effects
complex_user.sy,Full OOP,Without Side Effects
